# Asset Class Trend Following 策略回測 (equityNEW)

本策略採用 Asset Class Trend Following 方法，結合移動平均線 (SMA) 與 變動率 (ROC) 指標進行選股，並加入成交量濾網與停損機制。

In [ ]:
# 匯入所需套件
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle


## 資料清洗
從 Excel 檔案中讀取價格與成交量資料，處理空缺值並整理成 DataFrame。

In [ ]:
# 定義資料清洗函數
def clean_data(filepath):
    # 讀取 Excel 的兩個分頁
    df_prices = pd.read_excel(filepath, sheet_name='還原收盤價', header=None)
    df_volume = pd.read_excel(filepath, sheet_name='成交量', header=None)
    
    # 取得代號與名稱
    stock_codes = df_prices.iloc[0, 1:].values
    stock_names = df_prices.iloc[1, 1:].values
    
    # 整理日期
    date_strings = df_prices.iloc[2:, 0].astype(str).str[:8]
    dates = pd.to_datetime(date_strings, format='%Y%m%d')
    
    # 整理價格資料，處理空缺值 (前填補後填補)
    prices = df_prices.iloc[2:, 1:].astype(float)
    prices.index = dates
    prices.columns = stock_codes
    prices = prices.ffill().bfill()
    
    # 整理成交量資料，空缺補 0
    volumes = df_volume.iloc[2:, 1:].astype(float)
    volumes.index = dates
    volumes.columns = stock_codes
    volumes = volumes.fillna(0)
    
    code_to_name = dict(zip(stock_codes, stock_names))
    return prices, volumes, code_to_name

# 執行資料清洗
prices, volumes, code_to_name = clean_data('樣本集-1.xlsx')
print(f"資料筆數: {len(prices)}, 股票檔數: {len(prices.columns)}")


## 回測引擎
包含交易濾網、再平衡邏輯與多重停損機制。

In [ ]:
class BacktesterV2:
    def __init__(self, prices, volumes, code_to_name, initial_capital=30000000):
        self.prices_df = prices
        self.volumes_df = volumes
        self.prices = prices.values
        self.volumes = volumes.values
        self.dates = prices.index
        self.assets = prices.columns
        self.code_to_name = code_to_name
        self.initial_capital = initial_capital

    def run(self, sma_period, roc_period, stop_loss_pct, rebalance_interval=6, stop_loss_type='peak', ma_stop_period=5):
        # 1. 預計算指標 (SMA, ROC, 短期均線濾網)
        sma = self.prices_df.rolling(window=sma_period).mean().values
        roc = self.prices_df.pct_change(periods=roc_period).values
        sma5 = self.prices_df.rolling(window=5).mean().values
        sma10 = self.prices_df.rolling(window=10).mean().values
        sma20 = self.prices_df.rolling(window=20).mean().values
        
        ma_stop_vals = None
        if stop_loss_type == 'ma':
            ma_stop_vals = self.prices_df.rolling(window=ma_stop_period).mean().values

        # 2. 初始化帳戶與槽位 (3個槽位，各上限1000萬)
        surplus_pool = float(self.initial_capital)
        slots = {0: None, 1: None, 2: None}
        equity_curve_data = [] 
        trades_log = [] 
        holdings_history = [] 

        start_idx = max(sma_period, roc_period, 20)
        peak_equity = float(self.initial_capital)

        for i in range(start_idx, len(self.dates)):
            date = self.dates[i]
            current_prices = self.prices[i]

            # A. 計算權益
            stock_mv = 0.0
            for s_id, info in slots.items():
                if info:
                    stock_mv += info['shares'] * current_prices[info['asset_idx']]
            
            total_equity = surplus_pool + stock_mv
            peak_equity = max(peak_equity, total_equity)
            drawdown = (total_equity - peak_equity) / peak_equity
            equity_curve_data.append({'日期': date, '權益': total_equity, '回撤(Drawdown)': drawdown})

            if i == len(self.dates) - 1: break
            next_prices = self.prices[i+1] # T+1日執行價

            # B. 檢查停損
            for s_id, info in slots.items():
                if info:
                    a_idx = info['asset_idx']
                    curr_p = current_prices[a_idx]
                    triggered = False
                    if stop_loss_type == 'peak':
                        info['max_price'] = max(info['max_price'], curr_p)
                        if curr_p < info['max_price'] * (1 - stop_loss_pct): triggered = True
                    elif stop_loss_type == 'ma':
                        if curr_p < ma_stop_vals[i][a_idx]: triggered = True
                    
                    if triggered:
                        # 執行賣出
                        sell_p = next_prices[a_idx]
                        fee = info['shares'] * sell_p * 0.001425
                        tax = info['shares'] * sell_p * 0.003
                        surplus_pool += (info['shares'] * sell_p - fee - tax)
                        trades_log.append({'訊號日期': date, '標的': self.code_to_name[self.assets[a_idx]], '狀態': '賣出', '原因': '停損'})
                        slots[s_id] = None

            # C. 再平衡 (固定週期)
            if (i - start_idx) % rebalance_interval == 0:
                # 篩選符合條件標的 (均線向上、ROC>0、成交量濾網、短均線濾網)
                valid_signals = []
                sorted_idx = np.argsort(roc[i])[::-1]
                for idx in sorted_idx:
                    if len(valid_signals) >= 3: break
                    p = current_prices[idx]
                    amount = p * self.volumes[i][idx] * 1000
                    if p > sma[i][idx] and roc[i][idx] > 0 and amount > 30000000 and                        p > sma5[i][idx] and p > sma10[i][idx] and p > sma20[i][idx]:
                        valid_signals.append(idx)
                
                # 賣出不在名單中的持股
                current_assets = {info['asset_idx']: s_id for s_id, info in slots.items() if info}
                for a_idx, s_id in current_assets.items():
                    if a_idx not in valid_signals:
                        sell_p = next_prices[a_idx]
                        fee = slots[s_id]['shares'] * sell_p * 0.001425
                        tax = slots[s_id]['shares'] * sell_p * 0.003
                        proceeds = slots[s_id]['shares'] * sell_p - fee - tax
                        surplus_pool += proceeds
                        trades_log.append({'訊號日期': date, '標的': self.code_to_name[self.assets[a_idx]], '狀態': '賣出', '原因': '再平衡'})
                        slots[s_id] = {'pending_budget': proceeds}
                
                # 補足空槽位
                for a_idx in valid_signals:
                    if a_idx not in current_assets:
                        # 找尋可用槽位
                        for s_id in slots:
                            if slots[s_id] is None or 'pending_budget' in slots[s_id]:
                                budget = slots[s_id]['pending_budget'] if slots[s_id] else 10000000.0
                                if not slots[s_id]: surplus_pool -= budget
                                buy_p = next_prices[a_idx]
                                shares = (int(min(budget, 10000000.0) // (buy_p * 1.001425)) // 1000) * 1000
                                if shares > 0:
                                    cost = shares * buy_p * 1.001425
                                    surplus_pool += (budget - cost)
                                    slots[s_id] = {'asset_idx': a_idx, 'shares': shares, 'max_price': buy_p, 'budget': cost}
                                    trades_log.append({'訊號日期': date, '標的': self.code_to_name[self.assets[a_idx]], '狀態': '買進'})
                                else:
                                    surplus_pool += budget
                                    slots[s_id] = None
                                break

        return pd.DataFrame(equity_curve_data), pd.DataFrame(trades_log)

def calculate_metrics(eq_df):
    eq = eq_df['權益']
    total_ret = (eq.iloc[-1] / eq.iloc[0]) - 1
    years = (eq_df['日期'].iloc[-1] - eq_df['日期'].iloc[0]).days / 365.25
    cagr = (1 + total_ret)**(1/years) - 1
    mdd = eq_df['回撤(Drawdown)'].min()
    return cagr, mdd, cagr/abs(mdd)


## 執行回測與結果展示
使用最佳化後的參數進行回測 (SMA=117, ROC=59, SL=10%, Reb=9)。

In [ ]:
# 執行回測
bt = BacktesterV2(prices, volumes, code_to_name)
eq, trades = bt.run(117, 59, 0.1, 9, 'peak', 20)

# 計算績效 (2019-2023)
mask = (eq['日期'] >= '2019-01-01') & (eq['日期'] <= '2023-12-31')
cagr, mdd, calmar = calculate_metrics(eq[mask])

print(f"年化報酬率 (CAGR): {cagr:.2%}")
print(f"最大回撤 (MaxDD): {mdd:.2%}")
print(f"Calmar Ratio: {calmar:.2f}")

# 繪製權益曲線
plt.figure(figsize=(12, 6))
plt.plot(eq[mask]['日期'], eq[mask]['權益'])
plt.title('Equity Curve (2019-2023)')
plt.xlabel('Date')
plt.ylabel('Equity')
plt.grid(True)
plt.show()
